# 🎬 Colab ComfyUI Server

Auto-start ComfyUI + AnimateDiff on T4 GPU.

**Just click Runtime > Run all** — everything runs automatically.

After startup, the Tunnel URL is saved to Google Drive for your local client.


## Step 1: Check GPU

In [ ]:
!nvidia-smi


## Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
ROOT = "/content/drive/MyDrive/ComfyUI"
for d in ["models/checkpoints", "models/animatediff_models", "input", "output"]:
    os.makedirs(f"{ROOT}/{d}", exist_ok=True)
print("Drive ready:", ROOT)


## Step 3: Install ComfyUI

In [ ]:
import os
COMFY = "/content/workspace/ComfyUI"
if not os.path.exists(COMFY):
    !git clone --depth=1 https://github.com/comfyanonymous/ComfyUI.git {COMFY}
!cd {COMFY} && pip install -r requirements.txt -q --no-cache-dir
print("ComfyUI installed")


## Step 4: Install AnimateDiff + VideoHelper

In [ ]:
import os
N = f"{COMFY}/custom_nodes"
repos = {
    "ComfyUI-AnimateDiff-Evolved": "https://github.com/Kosinkadink/ComfyUI-AnimateDiff-Evolved.git",
    "ComfyUI-VideoHelperSuite": "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git",
}
for name, url in repos.items():
    p = f"{N}/{name}"
    if not os.path.exists(p):
        !git clone --depth=1 {url} {p}
        print(f"Installed {name}")
    else:
        print(f"Have {name}")
print("Nodes OK")


## Step 5: Download Models

In [ ]:
import os

CKPT_DIR = '/content/drive/MyDrive/ComfyUI/models/checkpoints'
AD_DIR = '/content/drive/MyDrive/ComfyUI/models/animatediff_models'
LOCAL_CKPT = '/content/workspace/ComfyUI/models/checkpoints'
LOCAL_AD = '/content/workspace/ComfyUI/models/animatediff_models'

os.makedirs(LOCAL_CKPT, exist_ok=True)
os.makedirs(LOCAL_AD, exist_ok=True)

def link_to_drive(drive_dir, local_dir, filename):
    drive_path = f'{drive_dir}/{filename}'
    local_path = f'{local_dir}/{filename}'
    if os.path.exists(drive_path) and not os.path.exists(local_path):
        !ln -s {drive_path} {local_path}
        return True
    return False

def download_if_missing(url, drive_dir, filename):
    path = f'{drive_dir}/{filename}'
    if not os.path.exists(path):
        print(f'Downloading {filename}...')
        !wget -q --show-progress "{url}" -O {path}
        print(f'    Done')
    else:
        print(f'    {filename} already in Drive')
    return path

# SD1.5 base model
print('SD1.5 base model:')
download_if_missing(
    'https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors',
    CKPT_DIR, 'v1-5-pruned-emaonly.safetensors'
)
link_to_drive(CKPT_DIR, LOCAL_CKPT, 'v1-5-pruned-emaonly.safetensors')

# Motion module v2
print('\nAnimateDiff motion module:')
download_if_missing(
    'https://huggingface.co/guoyww/animatediff/resolve/main/mm_sd_v15_v2.ckpt',
    AD_DIR, 'mm_sd_v15_v2.ckpt'
)
link_to_drive(AD_DIR, LOCAL_AD, 'mm_sd_v15_v2.ckpt')

print('\nAll models ready!')


## Step 6: Link I/O to Drive

In [ ]:
import os
for src, name in [("/content/drive/MyDrive/ComfyUI/output","output"),
                  ("/content/drive/MyDrive/ComfyUI/input","input")]:
    dst = f"/content/workspace/ComfyUI/{name}"
    if os.path.islink(dst) or os.path.isdir(dst):
        !rm -rf {dst}
    !ln -s {src} {dst}
print("I/O linked to Drive")


## Step 7: Start ComfyUI + Cloudflare Tunnel

In [ ]:
import subprocess, time, os, re, psutil, socket

def kill_process_on_port(port):
    for proc in psutil.process_iter(["pid", "name", "connections"]):
        try:
            for conn in proc.connections(kind="inet"):
                if conn.laddr.port == port:
                    print(f"Killing process {proc.pid} ({proc.name()}) on port {port}")
                    proc.terminate()
                    proc.wait(timeout=5)
        except Exception:
            pass

if not os.path.exists("/content/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
    !chmod +x /content/cloudflared

os.chdir("/content/workspace/ComfyUI")
kill_process_on_port(8188)

print("Starting ComfyUI...")
!rm -f /content/comfyui.log
os.system("nohup python main.py --listen 0.0.0.0 --port 8188 --lowvram > /content/comfyui.log 2>&1 &")

def wait_for_port(port, timeout=60):
    start_time = time.time()
    while time.time() - start_time < timeout:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            if sock.connect_ex(("127.0.0.1", port)) == 0:
                return True
        time.sleep(2)
    return False

if wait_for_port(8188):
    print("ComfyUI is active on port 8188.")
    print("Starting Cloudflared tunnel...")
    tunnel_proc = subprocess.Popen(
        ["/content/cloudflared", "tunnel", "--url", "http://localhost:8188"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )
    COMFY_URL = None
    for _ in range(30):
        line = tunnel_proc.stderr.readline().decode(errors="replace").strip()
        m = re.search(r"https://[\w.-]+\.trycloudflare\.com", line)
        if m:
            COMFY_URL = m.group(0)
            break
        time.sleep(1)
    if COMFY_URL:
        print(f"Tunnel URL: {COMFY_URL}")
        # Write tunnel URL to Drive for local client
        url_file = "/content/drive/MyDrive/ComfyUI/tunnel_url.txt"
        with open(url_file, "w") as f:
            f.write(COMFY_URL)
        print(f"Tunnel URL saved to Drive: {url_file}")
    else:
        print("ERROR: Failed to get tunnel URL")
else:
    print("ERROR: ComfyUI failed to start")


## Step 8: Server Ready - Waiting for Tasks
ComfyUI is running. Your local `gen_video.py` can now submit tasks.

This cell keeps the Colab session alive. Run it to prevent timeout.


In [ ]:
import time
print(f"Server ready at: {COMFY_URL}")
print("Waiting for tasks from local client...")
print("This cell keeps the session alive. Leave it running.")
print("Press the stop button to shut down.")
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("Server stopped.")
